# 第10章
## リスト10.1

In [ ]:
!pip install git+https://github.com/y0-causal-inference/y0.git@v0.2.0
from y0.dsl import P, Variable
E = Variable("E")
I = Variable("I")
query = P[E](I)
query

## リスト10.2

In [ ]:
!pip install graphviz==0.20.3
!apt install libgraphviz-dev
!pip install pygraphviz==1.13
!pip install networkx==2.8.8

In [ ]:
import requests
def download_code(url):
    response = requests.get(url)
    if response.status_code == 200:
        code_content = response.text
        print("Code fetched successfully.")
        return code_content
    else:
        print("Failed to fetch code.")
        return None
url = (
    "https://raw.githubusercontent.com/altdeep/"
    "causalAI/master/book/chapter%2010/id_utilities.py"
)
utilities_code = download_code(url)
print(utilities_code)

from y0.graph import NxMixedGraph as Y0Graph
from y0.dsl import P, Variable
G = Variable("G")
E = Variable("E")
I = Variable("I")
dag = Y0Graph.from_edges(
    directed=[
        (G, E),
        (G, I),
        (E, I)
    ]
)
gv_draw(dag)

## リスト10.3

In [ ]:
e = E
check_identifiable(
    dag,
    query=P(I @ e),
    distribution=P(G, E, I)
)

## リスト10.4

In [ ]:
check_identifiable(
    dag,
    query=P(I @ e),
    distribution=P(E, I)
)

## リスト10.5

In [ ]:
from y0.graph import NxMixedGraph as Y0Graph
from y0.dsl import P, Variable
from y0.algorithm.identify import Identification, identify

query=P(I @ e)
base_distribution = P(I, E, G)

identification_task = Identification.from_expression(
    graph=dag,
    query=query,
    estimand=base_distribution)

identify(identification_task)

## リスト10.6

In [ ]:
from y0.graph import NxMixedGraph as Y0Graph
from y0.dsl import P, Variable
G = Variable("G")
E = Variable("E")
I = Variable("I")
W = Variable("W")
e = E
dag = Y0Graph.from_edges(
    directed=[
        (G, E),
        (G, I),
        (E, W),
        (W, I)
    ]
)

query=P(I @ e)
base_distribution = P(I, E, W)

identification_task = Identification.from_expression(
    graph=dag,
    query=query,
    estimand=base_distribution)
identify(identification_task)

## リスト10.7

In [ ]:
T = Variable("T")
W = Variable("W")
B = Variable("B")
V = Variable("V")
C = Variable("C")
A = Variable("A")
t, a, w, v, b = T, A, W, V, B
dag = Y0Graph.from_edges(directed=[
    (T, W),
    (W, A),
    (B, V),
    (V, A),
    (C, T),
    (C, A),
    (C, B)
])
gv_draw(dag)

## リスト10.8

In [ ]:
from y0.algorithm.identify.idc_star import idc_star

idc_star(
    dag,
    outcomes={A @ -t: +a},
    conditions={T: +t}
)

## リスト10.9

In [ ]:
from y0.algorithm.identify.cg import make_parallel_worlds_graph
parallel_world_graph = make_parallel_worlds_graph(
    dag,
     {frozenset([+t])}
)
gv_draw(parallel_world_graph)

## リスト10.10

In [ ]:
from y0.algorithm.identify.cg import make_counterfactual_graph

events = {A @ -t: +a, T: +t}
cf_graph, _ = make_counterfactual_graph(dag, events)
gv_draw(cf_graph)

## リスト10.11

In [ ]:
parallel_world_graph = make_parallel_worlds_graph(
    dag,
   {frozenset([-t]), frozenset([-b])}
)
gv_draw(parallel_world_graph)

## リスト10.12

In [ ]:
joint_query = {A @ -t: +a, T: +t, B: -b, V @ -b: +v}
cf_graph, _ = make_counterfactual_graph(dag, joint_query)
gv_draw(cf_graph)

## リスト10.13

In [ ]:
!pip install numpyro==0.15.0
!pip install funsor==0.4.5
import jax.numpy as np
from jax import random
from numpyro import sample
from numpyro.handlers import condition, do
from numpyro.distributions import Bernoulli, Normal
from numpyro.infer import MCMC, NUTS
import matplotlib.pyplot as plt

rng = random.PRNGKey(1)

def model():
    p_member = 0.5
    is_guild_member = sample(
        "Guild Membership",
        Bernoulli(p_member)
    )
    p_engaged = (0.8*is_guild_member + 0.2*(1-is_guild_member))
    is_highly_engaged = sample(
        "Side-quest Engagement",
        Bernoulli(p_engaged)
    )
    p_won_engaged = (.9*is_highly_engaged + .1*(1-is_highly_engaged))
    high_won_items = sample("Won Items", Bernoulli(p_won_engaged))
    mu = (
        37.95*(1-is_guild_member)*(1-high_won_items) +
        54.92*(1-is_guild_member)*high_won_items +
        223.71*(is_guild_member)*(1-high_won_items) +
        125.50*(is_guild_member)*high_won_items
    )
    sigma = (
        23.80*(1-is_guild_member)*(1-high_won_items) +
        4.92*(1-is_guild_member)*high_won_items +
        5.30*(is_guild_member)*(1-high_won_items) +
        53.49*(is_guild_member)*high_won_items
    )
    in_game_purchases = sample("In-game Purchases", Normal(mu, sigma))

## リスト10.14

In [ ]:
intervention_model = do(model, {"Side-quest Engagement": np.array(0.)})
intervention_kernel = NUTS(intervention_model)
intervention_model_sampler = MCMC(
    intervention_kernel,
    num_samples=5000,
    num_warmup=200
)
intervention_model_sampler.run(rng)
intervention_samples = intervention_model_sampler.get_samples()
int_purchases_samples = intervention_samples["In-game Purchases"]

## リスト10.15

In [ ]:
cond_and_int_model = condition(
    intervention_model,
     {"Side-quest Engagement": np.array(1.)}
)
int_cond_kernel = NUTS(cond_and_int_model)
int_cond_model_sampler = MCMC(
    int_cond_kernel,
    num_samples=5000,
    num_warmup=200
)
int_cond_model_sampler.run(rng)
int_cond_samples = int_cond_model_sampler.get_samples()
int_cond_purchases_samples = int_cond_samples["In-game Purchases"]

## リスト10.16

In [ ]:
plt.hist(
    int_purchases_samples,
    bins=30,
    alpha=0.5,
    label='$P(I_{E=0})$'
)
plt.hist(
    int_cond_purchases_samples,
    bins=30,
    alpha=0.5,
    label='$P(I_{E=0}|E=1)$'
)
plt.legend(loc='upper left')
plt.show()